In [22]:
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf
# import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

pd.set_option("display.max_rows", None) # Mostrar todas las filas
pd.set_option("display.max_columns", None) # Mostrar todas las columnas
# pd.set_option("display.max_colwidth", None) # Mostrar el contenido completo de cada celda (sin truncar)
# pd.set_option("display.width", 0) # Ajustar el ancho máximo de la salida al 100% (evita cortes con "…")
# pd.set_option("display.precision", 10) # Opcional: mayor precisión

In [23]:
ticker = "PFAVAL.CL"
df = pd.read_csv(f"{ticker}.csv")
df.head()

,date,close,high,low,open,volume
0,2025-01-02,433.452301,433.452301,423.988277,422.095472,1242177
1,2025-01-03,438.184326,440.077131,434.398716,433.452314,2385850
2,2025-01-07,440.077118,440.077118,430.613094,438.184313,1979583
3,2025-01-08,438.184326,442.916338,435.345119,440.077131,564204
4,2025-01-09,435.345123,439.130733,435.345123,438.184331,3706319


# Modelo GRU (Gated Recurrent Unit)

## Parámetros del modelo

In [24]:
COLUMNA_OBJETIVO = "close" # Variable a predecir
lookback = 30 # Días de historia que ve el modelo
pred_len = 1 # Días hacia adelante a predecir
TEST_SIZE = 0.15 # 15 % de los datos para prueba
EPOCHS = 100
BATCH_SIZE = 16

In [25]:
import yfinance as yf

ticker = "PFAVAL.CL"
df = yf.download(ticker,
                 start="2025-01-01",
                 end="2026-05-28")


df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL
Date,,,,,
2025-01-02,433.452271,433.452271,423.988247,422.095442,1242177
2025-01-03,438.184296,440.077100,434.398686,433.452284,2385850
2025-01-07,440.077148,440.077148,430.613124,438.184343,1979583
2025-01-08,438.184296,442.916307,435.345089,440.077100,564204
2025-01-09,435.345093,439.130702,435.345093,438.184300,3706319


In [26]:
# ========================================================
# CLEAN DATA
# ========================================================

# Eliminar filas multi index
df.columns = df.columns.get_level_values(0)

# Verificar duplicados en el índice
duplicados = df.index.duplicated().sum()

print(f"Duplicados en índice: {duplicados}")

# Manejar nombres de columna
df.columns.name = None

df = df.reset_index()

df.columns = ['date','close','high','low','open','volume']

if df.empty:
    raise ValueError(f"No se encontraron datos historicos para {ticker}")


if "date" not in df.columns or COLUMNA_OBJETIVO not in df.columns:
    raise ValueError(
        f"El dataset descargado no contiene las columnas requeridas: date y {COLUMNA_OBJETIVO}"
    )

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").dropna(subset=[COLUMNA_OBJETIVO]).reset_index(drop=True)

if len(df) <= lookback + pred_len:
    raise ValueError(
        "No hay suficientes registros para entrenar el modelo con el lookback y pred_len solicitados"
    )

df.head()

Duplicados en índice: 0


,date,close,high,low,open,volume
0,2025-01-02,433.452271,433.452271,423.988247,422.095442,1242177
1,2025-01-03,438.184296,440.077100,434.398686,433.452284,2385850
2,2025-01-07,440.077148,440.077148,430.613124,438.184343,1979583
3,2025-01-08,438.184296,442.916307,435.345089,440.077100,564204
4,2025-01-09,435.345093,439.130702,435.345093,438.184300,3706319


In [27]:
csv_path = Path(f"{ticker}.csv")
csv_path

WindowsPath('PFAVAL.CL.csv')

## Preparar serie temporal

In [28]:
def preparar_datos(df, columna=COLUMNA_OBJETIVO, lookback=lookback):
    """Escala los precios y construye secuencias supervisadas (X, y)."""
    serie = df.sort_values("date")[[columna]].dropna().values # (N, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    serie_scaled = scaler.fit_transform(serie)

    X, y = [], []
    for i in range(lookback, len(serie_scaled) - pred_len + 1):
        X.append(serie_scaled[i - lookback : i])          # (lookback, 1)
        y.append(serie_scaled[i + pred_len - 1])        # (1,)

    X = np.array(X)   # (muestras, lookback, 1)
    y = np.array(y)   # (muestras, 1)
    return X, y, scaler, serie

# Dividir los datos en entrenamiento y prueba

In [29]:
def dividir_datos(X, y, test_size=TEST_SIZE):
    corte = int(len(X) * (1 - test_size))
    return X[:corte], X[corte:], y[:corte], y[corte:]

## Construir modelo GRU

In [30]:
def construir_modelo_gru(lookback=lookback):
    """Arquitectura GRU de dos capas (64 → 32 unidades) con Dropout + Dense para evitar sobreajuste."""
    modelo = Sequential([
    GRU(64, return_sequences=True, input_shape=(lookback, 1)),
    Dropout(0.2),
    GRU(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(pred_len)
    ])
    modelo.compile(optimizer="adam", loss="mean_squared_error")
    modelo.summary()
    return modelo

## Entrenar

In [31]:
def entrenar_modelo(modelo, X_train, y_train):
    """entrenamiento con EarlyStopping para cortar automáticamente cuando deja de mejorar."""
    early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    historia = modelo.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
    )
    return historia

## Evaluar y graficar

In [32]:
def evaluar_y_graficar(modelo, X_test, y_test, scaler, df):
    """calcula MAE y RMSE, y grafica real vs predicción"""
    pred_scaled = modelo.predict(X_test)
    pred = scaler.inverse_transform(pred_scaled)
    real = scaler.inverse_transform(y_test)
    mae  = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    print(f"\n📊 MAE  : {mae:.2f} COP")
    print(f"📊 RMSE : {rmse:.2f} COP")

    fechas_test = df.sort_values("date")["date"].values[
        -len(real) - pred_len + 1 : -pred_len + 1 if pred_len > 1 else None
    ]

    # Si vienen como (n, 1), los convertimos a 1D para graficar
    real_plot = real.ravel() if hasattr(real, "ravel") else real
    pred_plot = pred.ravel() if hasattr(pred, "ravel") else pred

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=fechas_test,
            y=real_plot,
            mode="lines",
            name="Precio real",
            line=dict(width=1.5)
        )
    )

    fig.add_trace(
        go.Scatter(
            x=fechas_test,
            y=pred_plot,
            mode="lines",
            name="Predicción GRU",
            line=dict(width=1.5, dash="dash")
        )
    )

    fig.update_layout(
        title="GRU - Predicción vs Precio real (PFAVAL)",
        xaxis_title="date",
        yaxis_title="Precio cierre (COP)",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
    )

    fig.show()
    return pred, real

## Predecir próximo día

In [33]:
def predecir_proximo_dia(modelo, df, scaler, columna=COLUMNA_OBJETIVO, lookback=lookback):
    """Toma los últimos lookback cierres y predice el siguiente precio."""
    ultimos = df.sort_values("date")[[columna]].dropna().values[-lookback:]
    ultimos_scaled = scaler.transform(ultimos).reshape(1, lookback, 1)
    pred_scaled = modelo.predict(ultimos_scaled, verbose=0)
    precio_futuro = scaler.inverse_transform(pred_scaled)[0, 0]
    ultima_fecha = df["date"].max()
    print(f"\n🔮 Precio proyectado para el siguiente día hábil "
    f"(desde {ultima_fecha.date()}): {precio_futuro:.2f} COP")
    return precio_futuro

In [34]:
"""
Ejecuta el pipeline completo sobre df y muestra la curva de aprendizaje, la gráfica de predicción vs real, 
y el precio proyectado para el siguiente día hábil.
"""

X, y, scaler, serie_original = preparar_datos(df)
X_train, X_test, y_train, y_test = dividir_datos(X, y)

print(f"Muestras entrenamiento : {len(X_train)}")
print(f"Muestras prueba : {len(X_test)}")

modelo_gru = construir_modelo_gru()
historia = entrenar_modelo(modelo_gru, X_train, y_train)

# Curva de pérdida
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=list(range(1, len(historia.history["loss"]) + 1)),
        y=historia.history["loss"],
        mode="lines",
        name="Loss entrenamiento",
        line=dict(width=1.8)
    )
)

fig.add_trace(
    go.Scatter(
        x=list(range(1, len(historia.history["val_loss"]) + 1)),
        y=historia.history["val_loss"],
        mode="lines",
        name="Loss validación",
        line=dict(width=1.8, dash="dash")
    )
)

fig.update_layout(
    title="Curva de aprendizaje GRU",
    xaxis_title="Época",
    yaxis_title="MSE",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    height=350
)

fig.show()

predicciones, reales = evaluar_y_graficar(modelo_gru, X_test, y_test, scaler, df)
predecir_proximo_dia(modelo_gru, df, scaler)

c:\Users\Gisela\Documents\GitHub\Talleres_IA_2026\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Muestras entrenamiento : 272
Muestras prueba : 48


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 30, 64)         │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,817 (89.13 KB)

...

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 0.0773 - val_loss: 0.0222
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0127 - val_loss: 0.0059
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0093 - val_loss: 0.0044
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0074 - val_loss: 0.0058
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0060 - val_loss: 0.0075
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0048 - val_loss: 0.0050
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0047 - val_loss: 0.0049
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0039 - val_loss: 0.0059
Epoch 9/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0041 - val_loss: 0.0050
Epoch 10/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0044 - val_loss: 0.0050
Epoch 11/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0043 - val_loss: 0.0038
Epoch 12/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step

📊 MAE  : 19.31 COP
📊 RMSE : 24.55 COP



🔮 Precio proyectado para el siguiente día hábil (desde 2026-05-27): 831.31 COP


np.float32(831.30615)

In [39]:
predecir_proximo_dia(modelo_gru, df, scaler, lookback=7)


🔮 Precio proyectado para el siguiente día hábil (desde 2026-05-27): 830.31 COP


np.float32(830.30774)

# Gráfica de velas

In [36]:
# Soportes y resistencias + velas japonesas usando df (Plotly)

def detectar_soportes_resistencias(
    df, lookback=3, tolerancia_pct=0.007, max_niveles=6,
    graficar=False, ultimas_barras=120
 ):
    """
    Detecta niveles por pivotes locales y agrupa precios cercanos.
    - lookback: barras a izquierda/derecha para definir pivote
    - tolerancia_pct: umbral de agrupacion (0.007 = 0.7%)
    - graficar: si True, muestra velas + niveles con Plotly
    """
    data = df.sort_values("date").reset_index(drop=True).copy()

    lows = data["low"].values
    highs = data["high"].values

    soportes_raw = []
    resistencias_raw = []

    for i in range(lookback, len(data) - lookback):
        low_win = lows[i - lookback : i + lookback + 1]
        high_win = highs[i - lookback : i + lookback + 1]

        if lows[i] == low_win.min():
            soportes_raw.append(float(lows[i]))
        if highs[i] == high_win.max():
            resistencias_raw.append(float(highs[i]))

    def agrupar_niveles(niveles):
        if not niveles:
            return []
        niveles = sorted(niveles)
        grupos = [[niveles[0]]]

        for nivel in niveles[1:]:
            media_grupo = sum(grupos[-1]) / len(grupos[-1])
            if abs(nivel - media_grupo) / media_grupo <= tolerancia_pct:
                grupos[-1].append(nivel)
            else:
                grupos.append([nivel])

        # Centro de cada grupo
        centros = [sum(g) / len(g) for g in grupos]
        return centros

    soportes = agrupar_niveles(soportes_raw)
    resistencias = agrupar_niveles(resistencias_raw)

    # Nos quedamos con niveles mas representativos en el rango de precios
    cierre_min = float(data["close"].min())
    cierre_max = float(data["close"].max())

    soportes = [s for s in soportes if s <= cierre_max]
    resistencias = [r for r in resistencias if r >= cierre_min]

    # Limitar cantidad para no saturar la grafica
    soportes = sorted(soportes)[-max_niveles:]
    resistencias = sorted(resistencias)[:max_niveles]

    if graficar:
        data_plot = data.tail(ultimas_barras).copy()
        data_plot["open"] = data_plot["close"].shift(1)
        data_plot.iloc[0, data_plot.columns.get_loc("open")] = data_plot.iloc[0]["close"]

        fig = go.Figure(
            data=[
                go.Candlestick(
                    x=data_plot["date"],
                    open=data_plot["open"],
                    high=data_plot["high"],
                    low=data_plot["low"],
                    close=data_plot["close"],
                    name="Velas"
                )
            ]
        )

        for s in soportes:
            fig.add_hline(y=s, line=dict(color="#1f77b4", width=1.1, dash="dash"), opacity=0.85)
        for r in resistencias:
            fig.add_hline(y=r, line=dict(color="#ff7f0e", width=1.1, dash="dash"), opacity=0.85)

        fig.update_layout(
            title="PFAVAL - Velas japonesas con soportes y resistencias",
            yaxis_title="Precio (COP)",
            xaxis_title="date",
            xaxis_rangeslider_visible=False,
            template="plotly_white",
            height=650
        )
        fig.show()

    return soportes, resistencias


# Ejecutar sobre df
soportes, resistencias = detectar_soportes_resistencias(
    df,
    lookback=3,
    tolerancia_pct=0.007,
    max_niveles=6,
    graficar=True,
    ultimas_barras=120
)

print("Soportes detectados:", [round(s, 2) for s in soportes])
print("Resistencias detectadas:", [round(r, 2) for r in resistencias])

Soportes detectados: [738.49, 748.58, 754.69, 761.88, 770.82, 790.48]
Resistencias detectadas: [442.92, 551.72, 568.77, 575.5, 605.28, 627.16]


In [37]:
df.tail()

,date,close,high,low,open,volume
345,2026-05-21,777.390137,782.373407,764.433634,776.393483,610851
346,2026-05-22,776.393494,785.363380,764.433645,777.390148,656376
347,2026-05-25,789.349976,791.343284,771.410203,776.393473,722432
348,2026-05-26,844.000000,844.000000,790.000000,792.000000,2450232
349,2026-05-27,840.000000,852.000000,834.000000,844.000000,2742929
